Spring Kafka

## Producer
### Producer Factory
Is an abstraction that creates and manages `KafkaProducer` instances. Its interface looks like:

In [ ]:
public interface ProducerFactory<K, V> {
    KafkaProducer<K, V> createProducer();
    
    String getTransactionIdPrefix();
    
    void closeProducer(KafkaProducer<K, V> producer);
    
    default boolean transactionIdPrefixSupplied() { ... }
}

`DefaultKafkaProducerFactory` is the default implementation which accepts a map containing configurations:

In [ ]:
new DefaultKafkaProducerFactory<String, String>(
        Map.of(
                ProducerConfig.BOOTSTRAP_SERVERS_CONFIG, "localhost",
                ProducerConfig.RETRIES_CONFIG, 3,
                ProducerConfig.KEY_SERIALIZER_CLASS_CONFIG, StringSerializer.class,
                ProducerConfig.VALUE_SERIALIZER_CLASS_CONFIG, StringSerializer.class,
                ProducerConfig.ACKS_CONFIG, "all"
        )
);

Since `KafkaProducer` is thread safe multiple threads can share the same instance. Thus, when not using transactions, `DefaultKafkaProducerFactory` creates a singleton producer. There is facility to create (and cache) new producer instance per thread.

### KafkaTemplate
`KafkaTemplate` wraps a producer and provides convenience methods to send data to Kafka topics. To initialise, pass a `ProducerFactory` instance:

In [ ]:
new KafkaTemplate<String, String>(producerFactory);

Some of the most used methods to send message are:

In [ ]:
// Sending messages
CompletableFuture<SendResult<String, String>> future = kafkaTemplate.send(TOPIC, "India", "New Delhi"); // At this point message is not necessary to be sent over network
future.whenComplete((result, ex) -> {
    if (ex != null) {
        System.err.println("Error occurred while sending message");
    } else {
        RecordMetadata metadata = result.getRecordMetadata();
        System.out.println(metadata.topic());
        System.out.println(metadata.partition());
        System.out.println(metadata.offset());
    }
});

// Blocking variant
SendResult<String, String> result = kafkaTemplate.send(TOPIC, "Germany", "Berlin").get();

// Some other variants
sendDefault("Japan", "Tokyo"); // sends to default topic
send(TOPIC, 0, "Canada", "Ottawa"); // send to specific partition

// Message variant
Message<String> message = MessageBuilder
    .withPayload("Moscow")
    .setHeader(KafkaHeaders.TOPIC, TOPIC)
    .setHeader(KafkaHeaders.KEY, "Russia")
    .build();
kafkaTemplate.send(message);

### Transactional Producer

## Consumer
### Consumer Factory
Abstraction that creates and manages `KafkaConsumer` instances. Its interface looks like:

In [ ]:
public interface ConsumerFactory<K, V> {
    // Creates consumers using the client id and group id configured in properties
	default Consumer<K, V> createConsumer() { ... }

    // Creates consumers using the  provided client id suffix
	default Consumer<K, V> createConsumer(String clientIdSuffix) { ... }

    // Creates consumers using the  provided client id and group id
	default Consumer<K, V> createConsumer(String groupId, String clientIdSuffix) { ... }

/*
Client id is mainly used for:
- logging
- metrics
- monitoring
- quotas
- tracing/debugging

It does NOT affect:
- partition assignment
- consumer group membership
- offsets
*/

`DefaultKafkaConsumerFactory` is the default implementation:

In [ ]:
return new DefaultKafkaConsumerFactory<String, String>(
        Map.of(
                ConsumerConfig.BOOTSTRAP_SERVERS_CONFIG, "localhost",
                ConsumerConfig.GROUP_ID_CONFIG, "my-test-consumer-group",
                ConsumerConfig.KEY_DESERIALIZER_CLASS_CONFIG, StringDeserializer.class,
                ConsumerConfig.VALUE_DESERIALIZER_CLASS_CONFIG, StringDeserializer.class,
                ConsumerConfig.ENABLE_AUTO_COMMIT_CONFIG, true
        )
);

### MessageListenerContainer
As we know, the way to read messages from Kafka is by invoking poll inside a infinite while loop. Spring abstracts away this in `MessageListenerContainer`. There are two implementations provided:
- `KafkaMessageListenerContainer`
- `ConcurrentMessageListenerContainer`

`KafkaMessageListenerContainer` does:
1. Uses `ConsumerFactory` instance to create `KafkaConsumer` instance on initialization
2. Subscribes to specified topics
3. Sets up `poll` loop
4. Delegates messages to `MessageListener` (typically the bean method annotated with `@KafkaListener`)
5. Manages the commit phase based on the configured `AckMode`:
    1. `RECORD`: Commits the offset after each individual record is processed.
    2. `BATCH`: Commits the offset after the entire batch is successfully processed.
    3. `MANUAL`/`MANUAL_IMMEDIATE`: Requires the developer to manually trigger the `Acknowledgment.acknowledge()` method.
6. Delegates to appropriate `ErrorHandler` when error happens while consuming message.

Note that there is a auto-commit property set at `ConsumerFactory` level. And there is `AckMode` set at `MessageListenerContainer` level. It is advised to set the former as false and let the listener container handle offset commit once messages are processed by the message listeners.

**Concurrency:** `KafkaMessageListenerContainer` is single threaded, it consumes and processes messages one at a time. To increase message processing parallelism, we can create more `KafkaConsumer` instances and the parititions would be distributed accordingly. This is what `ConcurrentMessageListenerContainer` does.

In [ ]:
@Bean
public ConcurrentKafkaListenerContainerFactory<String, String> kafkaListenerContainerFactory() {
    ConcurrentKafkaListenerContainerFactory<String, String> factory = 
        new ConcurrentKafkaListenerContainerFactory<>();
    
    factory.setConsumerFactory(consumerFactory());
    factory.setConcurrency(3); // 3 consumer threads, 3 KafkaConsumer instances, 3 poll loops
                               // Concurrency number should be set max at total number of partitions    
    return factory;
}

### MessageListener
When using `MessageListenerContainer`, we need to set up `MessageListener` instances which would actually receive messages from `MessageListenerContainer`. The interface looks like:

In [ ]:
public interface MessageListener<K, V> { 
    void onMessage(ConsumerRecord<K, V> data);
}

This message listener is passed on to a `MessageListenerContainer`:

In [ ]:
ContainerProperties containerProperties = new ContainerProperties("my-topic");
containerProperties.setAckMode(ContainerProperties.AckMode.RECORD);

KafkaMessageListenerContainer<String, String> messageListenerContainer = 
    new KafkaMessageListenerContainer<>(c, containerProperties);
messageListenerContainer.setupMessageListener((MessageListener<String, String>) message -> {
    System.out.println("Message key is " + message.key() + " value is " + message.value() 
            + " received from " + message.topic() + " partition " + message.partition());
}); 

There is a variant available which lets us perform manual offset committing using `Acknowledgment` instance.

In [ ]:
public interface AcknowledgingMessageListener<K, V> extends MessageListener<K, V> {
	@Override
	void onMessage(ConsumerRecord<K, V> data, Acknowledgment acknowledgment);

    // ...
}

The more common way to setup message listeners is using `@KafkaListener` annotation. A typical implementation looks like:

In [ ]:
@KafkaListener(topics = "orders")
public void consume(String message) { // Value part of ConsumerRecord
    System.out.println("Message: " + message);
}

@KafkaListener(topics = {"orders", "payment"}) // Can read from multiple topics
public void consume(ConsumerRecord<String, String> record) {
    String key = record.key();
    String value = record.value();
    long offset = record.offset();
    int partition = record.partition();
    long timestamp = record.timestamp();
    
    // ...
}

// Other variants

A more elaborate example:

In [ ]:
@Service
@Slf4j
public class EnterpriseOrderConsumer {
    
    @Autowired
    private OrderService orderService;
    
    @Autowired
    private KafkaTemplate<String, String> kafkaTemplate;
    
    @Autowired
    private MetricsService metricsService;
    
    // High-priority orders: high concurrency
    @KafkaListener(
        id = "priority-orders",  // Listener id, local concept to Spring Kafka
        topics = "orders-priority",
        groupId = "order-service-priority",  // overriding consumer group
        containerFactory = "highConcurrencyFactory"  // set a custom MessageListenerContainerFactory
    )
    public void consumePriorityOrders(
        Order order,
        ConsumerRecord<String, Order> record,
        Acknowledgment acknowledgment) {  // For manual offset commit
        
        long startTime = System.currentTimeMillis();
        
        try {
            log.info("Processing priority order: {}, partition: {}, offset: {}",
                order.getId(), record.partition(), record.offset());
            
            orderService.processPriorityOrder(order);
            
            acknowledgment.acknowledge();
            metricsService.recordSuccess("priority-orders", startTime);
            
        } catch (ValidationException e) {
            log.error("Validation failed for order: {}", order.getId(), e);
            kafkaTemplate.send("orders-dlq", order.getId(), order.toString());  // Dead letter queue
            acknowledgment.acknowledge(); // Skip
            metricsService.recordFailure("priority-orders", "validation");
        } catch (Exception e) {
            log.error("Unexpected error processing order: {}", order.getId(), e);
            metricsService.recordFailure("priority-orders", "error");
            throw e; // Retry
        }
    }
    
    // Regular orders: lower concurrency
    @KafkaListener(
        id = "regular-orders",
        topics = "orders-regular",
        groupId = "order-service-regular",
        containerFactory = "lowConcurrencyFactory"
    )
    public void consumeRegularOrders(List<Order> orders) {  // Batch processing variant
        log.info("Batch processing {} regular orders", orders.size());
        
        try {
            orderService.processBatchOrders(orders);
        } catch (Exception e) {
            log.error("Batch processing failed", e);
            throw e;
        }
    }
    
    // Dead-letter queue handler
    @KafkaListener(
        topics = "orders-dlq",
        groupId = "order-service-dlq"
    )
    public void handleDeadLetters(ConsumerRecord<String, String> record) {
        log.error("Dead letter received. Key: {}, Value: {}", 
            record.key(), record.value());
        
        // Send alert
        metricsService.recordDeadLetter("orders");
    }
}

To use `@KafkaListener` we need to provide a bean of `KafkaListenerContainerFactory` which would create `KafkaListenerContainer` instances as required:

In [ ]:
@Bean
public ConcurrentKafkaListenerContainerFactory<String, String>
kafkaListenerContainerFactory() {

    ConcurrentKafkaListenerContainerFactory<String, String> factory =
            new ConcurrentKafkaListenerContainerFactory<>();

    factory.setConsumerFactory(consumerFactory());

    return factory;
}

Error handlers can also be associated:

In [ ]:
@Bean
public CommonErrorHandler errorHandler() {
    DefaultErrorHandler handler = new DefaultErrorHandler((record, exception) -> {
        System.err.println("Error processing record: " + record);
        System.err.println("Exception: " + exception.getMessage());
        
        // Send to dead-letter topic
        kafkaTemplate.send("orders.dead-letter", 
                          record.key(), 
                          record.value());
    }, 
    new FixedBackOff(1000, 3)); // Retry 3 times with 1s delay
    
    return handler;
}

ConcurrentKafkaListenerContainerFactory<String, String> factory = 
        new ConcurrentKafkaListenerContainerFactory<>();
    
factory.setConsumerFactory(consumerFactory());
factory.setCommonErrorHandler(errorHandler());